In [ ]:
# for root anchor (notebook moved into subfolder)
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import plotly.graph_objects as go

import paths
from augmentation_pipeline import AugmentationConfig, FullRangeDataAugmentor, SignalAugmentor
from plot_style import apply_default_plotly_layout

# Replicate labels for the calibration workbook columns - same list used by
# data_registry.CONCENTRATIONS_uM / full_range_data_augmentation.ipynb.
CONCENTRATIONS_uM = [
    100, 100, 100, 50, 50, 50, 25, 25, 25, 15, 15, 15,
    10, 10, 10, 7.5, 7.5, 7.5, 5, 5, 5, 2.5, 2.5,
    1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
    0.25, 0.25, 0.25, 0.1, 0.1, 0.1, 0.1, 0.1,
]

# real pipeline
augmentor = FullRangeDataAugmentor.from_excel(paths.CALIBRATION_WORKBOOK, concentrations_uM=CONCENTRATIONS_uM)
calibration = augmentor.calibration
E = calibration.potential_grid_V

config = AugmentationConfig()
signal_augmentor = SignalAugmentor(calibration, augmentor.ip_spline, augmentor.noise_model, config)

C_TARGET = 6.0 

# pchip
c_lo, I_lo, c_hi, I_hi = signal_augmentor.get_base_pair(C_TARGET)
I_base = signal_augmentor.interpolate_base_curve(C_TARGET, c_lo, I_lo, c_hi, I_hi)


In [ ]:
# Graph 1 - Interpolare PCHIP between anchor curves
#
# SignalAugmentor.get_base_pair() + interpolate_base_curve() din augmentation_pipeline.py:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=E, y=I_lo,
    mode='lines', name=f'Curbă ancoră reală @ {c_lo:g} µM',
    line=dict(color='#7f7f7f', width=2, dash='dot')
))
fig.add_trace(go.Scatter(
    x=E, y=I_hi,
    mode='lines', name=f'Curbă ancoră reală @ {c_hi:g} µM',
    line=dict(color='#1f77b4', width=2, dash='dot')
))
fig.add_trace(go.Scatter(
    x=E, y=I_base,
    mode='lines', name=f'Interpolare PCHIP @ {C_TARGET:g} µM',
    line=dict(color='#2ca02c', width=3)
))

fig = apply_default_plotly_layout(
    fig,
    title_text='Interpolarea PCHIP între curbele ancoră',
    xaxis_title='Potențial E (V)',
    yaxis_title='Curent I (µA)'
    
)
fig.update_xaxes(range=[-0.6, 0.2])
fig.show()


In [ ]:
# Graph 2 - polynomial baseline distortion
#
# SignalAugmentor.apply_polynomial_baseline_distortion() din augmentation_pipeline.py
# SCALED: baseline_amp_max_uA (0.05 -> 3.0) for visibility
demo_config_baseline = AugmentationConfig(baseline_amp_max_uA=3.0)
demo_augmentor_baseline = SignalAugmentor(calibration, augmentor.ip_spline, augmentor.noise_model, demo_config_baseline)

np.random.seed(106)
I_distorted = demo_augmentor_baseline.apply_polynomial_baseline_distortion(I_base.copy(), C_TARGET)
distortion = I_distorted - I_base

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=E, y=I_base,
    mode='lines', name='Semnal original (interpolat)',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=E, y=distortion,
    mode='lines', name='Distorsiune generată',
    line=dict(color='#d62728', width=2, dash='dash')
))
fig.add_trace(go.Scatter(
    x=E, y=I_distorted,
    mode='lines', name='Semnal distorsionat',
    line=dict(color='#2ca02c', width=2)
))

fig = apply_default_plotly_layout(
    fig,
    title_text='Distorsiune polinomială de baseline (scalat pentru vizibilitate)',
    xaxis_title='Potențial E (V)',
    yaxis_title='Curent I (µA)'
)
fig.update_xaxes(range=[-0.6, 0.2])
fig.show()


In [ ]:
# Graph 3 potential drift orizontal
#
# different seed for visible shift
demo_config_drift = AugmentationConfig()
demo_augmentor_drift = SignalAugmentor(calibration, augmentor.ip_spline, augmentor.noise_model, demo_config_drift)

np.random.seed(56)
I_drifted = demo_augmentor_drift.apply_horizontal_potential_drift(I_base.copy())

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=E, y=I_base,
    mode='lines', name='Semnal original (interpolat)',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=E, y=I_drifted,
    mode='lines', name='Semnal cu potential drift',
    line=dict(color='#ff7f0e', width=2)
))

fig = apply_default_plotly_layout(
    fig,
    title_text='Drift de potential (potential drift orizontal)',
    xaxis_title='Potențial E (V)',
    yaxis_title='Curent I (µA)'
)
fig.update_xaxes(range=[-0.6, 0.2])
fig.show()


In [ ]:
# Graph4 - Efectul zgomotului Long & Winefordner
#
# SignalAugmentor.apply_position_dependent_noise() din augmentation_pipeline.p
# real noise
np.random.seed(42)
I_noisy = signal_augmentor.apply_position_dependent_noise(I_base.copy(), C_TARGET)

fig = go.Figure()

fig.add_vrect(
    x0=config.E_peak_start, x1=config.E_peak_end,
    fillcolor='#7f7f7f', opacity=0.12, line_width=0,
    annotation_text='Fereastră de vârf', annotation_position='top left'
)
fig.add_trace(go.Scatter(
    x=E, y=I_base,
    mode='lines', name='Semnal original (interpolat)',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=E, y=I_noisy,
    mode='lines', name='Semnal cu zgomot L&W',
    line=dict(color='#9467bd', width=1.5)
))

fig = apply_default_plotly_layout(
    fig,
    title_text='Zgomot Long & Winefordner dependent de poziție',
    xaxis_title='Potențial E (V)',
    yaxis_title='Curent I (µA)'
)
fig.update_xaxes(range=[-0.6, 0.2])
fig.show()
